# Chapter 5: Re-ranking

**Estimated time: 15 minutes**

Your chunking is clean. Your query rewriter is honest. Your retrieval routes the question to the right index. And your vector search still puts the wrong document on top. The score looks fine. The words overlap. The answer is still wrong.

In this notebook you will watch a real user question where the correct answer sits buried at rank four of the initial vector search. You will see the LLM confidently give the wrong answer because the right chunk never reaches it. Then you will add one function call, Cohere Rerank, and watch the correct chunk jump to rank one and the answer flip from wrong to right. Same corpus. Same question. Same LLM. What changes is the order of the chunks that get handed to the model.

## What you will see in three minutes

You are going to ask this question, the kind of thing a frustrated customer actually types into a support chat:

> If I do not like Pro Annual any more, how do I get my money back from the yearly purchase?

The correct answer lives in the official refund policy, Section 4.2. Pro Annual refunds are pro-rated, minus two months of service, minus a fifty dollar processing fee. Short, specific, and unambiguous.

You will run this through two versions of the same RAG pipeline.

- **Version A (vector only).** FAISS returns the top three chunks by cosine similarity. The top chunk is a billing FAQ snippet about switching plans. The second is the cancellation process section. The third is an outdated blog post from 2024 that announces a thirty day no-questions-asked full refund that no longer exists. The correct pro-rated calculation sits at rank four, never reaches the LLM, and the LLM answers with the thirty day full refund from the stale blog post. Confidently. Wrongly.
- **Version B (vector plus Cohere Rerank).** FAISS still returns ten candidates. Cohere Rerank reads every candidate paired with the question and re-sorts them. The pro-rated calculation chunk jumps from rank four to rank two with a Cohere relevance score near 0.97. The refund policy dominates the new top three. The LLM now sees the real pro-rated formula as its primary context and returns the correct math.

One function call separates a production incident from a correct answer. That is the reranking lesson.

Watch the rank column and the relevance column as you work through the notebook. The rank change is what the user would see on a support page. The relevance score is how confident the cross-encoder is that the document actually answers the question, not just that the words overlap.

## Install the dependencies

Run the next cell once. In Colab it installs the Python packages fresh. Locally it assumes you already ran `pip install -r requirements.txt` in your virtual environment.

Chapter 5 adds one new dependency to the list, `cohere`. Every earlier chapter works without it. Starting now, `cohere` is required.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected. Cleaning up any prior chromadb install (silent)...")
    !pip uninstall -y -q chromadb chroma-hnswlib langchain-chroma 2>/dev/null

    print("Installing dependencies (takes about 60 seconds)...")
    !pip install -q \
        langchain==0.3.14 \
        langchain-openai==0.2.14 \
        langchain-community==0.3.14 \
        langchain-text-splitters==0.3.5 \
        faiss-cpu==1.13.2 \
        pypdf==5.1.0 \
        langsmith==0.2.6 \
        pandas==2.2.2 \
        matplotlib==3.9.4 \
        cohere==5.13.3
    print("Done.")
else:
    print("Local environment detected. Skipping pip install.")
    print("Make sure you have run 'pip install -r requirements.txt' in your venv.")

In [ ]:
import os, sys

if IN_COLAB:
    # Always force a fresh clone so updates on GitHub are picked up. Without
    # this, a stale repo from a prior session would keep running the old utils.
    !rm -rf rag-for-pms
    !git clone -q https://github.com/DDRXV/rag-for-pms.git
    os.chdir("rag-for-pms")
else:
    # Local Jupyter: already inside the repo. Walk up to root if we are in chapters/.
    if os.path.basename(os.getcwd()) == "chapters":
        os.chdir("..")

# Python caches imported modules in sys.modules. Drop any cached utils.* so
# the next import reads the freshly cloned file, not a stale copy from an
# earlier session.
for cached in [m for m in sys.modules if m.startswith("utils")]:
    del sys.modules[cached]

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

In [ ]:
from utils.shared import get_keys
get_keys()

## What you just did

`get_keys` pulled your OpenAI, LangSmith, and Cohere keys out of Colab secrets (you set these up in Chapter 1) and turned on LangSmith tracing for this notebook. Chapter 5 is the first chapter that actually calls the Cohere API. If you skipped adding `COHERE_API_KEY` to secrets back in Chapter 1, now is the time to add it. Every rerank cell below needs it.

## Look at the corpus

The corpus is almost the same six SkillAgents AI documents from earlier chapters. Pricing, product guide, billing FAQ, error codes, refund policy, and one outdated blog post. This chapter adds a seventh file, `ch5_refund_quick_answers.md`, which is a short community reference page written by the student success team. It points customers at the refund policy but deliberately does not contain any refund numbers of its own. It is there to make the retrieval problem more realistic, because in a real production corpus you always have multiple documents that mention the same topic at different levels of specificity.

In [ ]:
from utils.shared import load_corpus

docs = load_corpus()
for d in docs:
    print(f"  {d.metadata.get('source', 'unknown'):32s}  {len(d.page_content):5d} chars")

## Peek at the target passage

Before we run anything, let us look at the exact text that answers the question. It lives inside `refund_policy.pdf` at Section 4.2, and it is the only place in the corpus that spells out the pro-rated refund math for Pro Annual.

In [ ]:
refund = next(d for d in docs if d.metadata["source"] == "refund_policy.pdf")

start = refund.page_content.find("4.2")
print(refund.page_content[start:start + 650])

This is the chunk we want the LLM to see. Read it carefully. It says the refund is pro-rated, it says two months of service are deducted, and it says there is a fifty dollar processing fee. A worked example calculates $824.42 on a $1,499 plan canceled five months in. That is the ground truth for the rest of the notebook.

Now let us see what the vector search actually retrieves.

## Build the index

Use the same chunking settings Chapter 1 landed on. `chunk_size=500` and `chunk_overlap=50`. Every retrieval experiment in this chapter runs against the same index, so the only thing changing is what happens after retrieval.

In [ ]:
from utils.shared import build_index

index = build_index(chunk_size=500, chunk_overlap=50)

# A real user question, phrased the way a frustrated customer would type it.
# The words "not like", "money back", and "yearly purchase" are casual and
# do not match the specific vocabulary of the refund policy document.
question = "If I do not like Pro Annual any more, how do I get my money back from the yearly purchase?"
print(f"Question: {question}")

## Version A: vector search only, the naive pipeline

First the default. Ask FAISS for the top ten chunks by cosine similarity. We ask for ten, not three, because we want to see where the correct chunk actually lives in the ranking, not just whether it made the top three. The chunks that end up in the LLM prompt are the top three by default.

In [ ]:
from utils.shared import search, show_results

vector_results = search(index, question, k=10)
show_results(vector_results, question=question)

## Read the table carefully

This is where the lesson lives. Walk through the rank column from the top.

- **Rank 1** is a billing FAQ chunk. The snippet talks about switching from Student to Pro Annual, which is not a refund at all. The vector search picked it because "Pro Annual" and "money" appear in the same sentence.
- **Rank 2** is the refund policy cancellation process, Section 4.1. It describes how to cancel Pro Annual but explicitly says cancellation on its own does not issue a refund.
- **Rank 3** is `outdated_blog_post.md`. This is the 2024 announcement of a thirty day full refund guarantee that no longer exists. The words overlap heavily with the question, so the embedding thinks it is a great match. It is not. The policy changed years ago.
- **Rank 4** is finally the correct chunk from the refund policy, the pro-rated calculation from Section 4.2. This is the chunk that holds the real answer. It is behind three decoys.

A real production RAG pipeline almost always sends the top three chunks to the LLM, not all ten. So this pipeline is about to hand the LLM the billing FAQ, the cancellation process, and the stale blog post. The correct chunk never makes it past the retrieval step.

Watch what that does to the answer.

In [ ]:
from utils.shared import generate_answer

# A real production RAG pipeline usually feeds the top three chunks to the LLM,
# so do exactly that. No filtering, no rerank. Just the top of the vector search.
vector_top_3 = vector_results[:3]
answer_vector_only = generate_answer(vector_top_3, question)
print(answer_vector_only)

## What just happened

Read that answer. The LLM is telling the customer they can get a thirty day full refund with no deductions. That was true in 2024 when the blog post was written. It is not true in the current policy, which is Section 4.2 of `refund_policy.pdf`. The pipeline took the stale blog post at face value because it was in the top three.

This is the failure mode reranking exists to fix. The retrieval scores looked reasonable. The cosine similarity on the blog post chunk was around 0.56, which lands inside the "decent match" band on the scale Chapter 1 established. The embedding model cannot tell which document reflects current policy and which one is stale. It only compares similarity in vector space. A close match on a stale document is still a close match.

The fix is not to throw away the stale document or to tune the similarity threshold. The fix is a second stage that reads each candidate paired with the question and asks whether it actually answers the question. That is what a cross-encoder does. That is what Cohere Rerank does.

## Bi-encoders vs cross-encoders in one paragraph

The embedding model behind FAISS is a bi-encoder. It processes the question and every document separately, turns each one into a vector, and compares the two vectors with cosine similarity. This is fast, because you can pre-compute every document embedding once and store it. At query time you only encode the question and do vector math. Millions of documents, millisecond latency. The tradeoff is that the encoder never sees the question and the document together. It cannot reason about how they relate. It just compares two points in space.

A cross-encoder takes a different approach. It concatenates the question and a single document into one input and runs them through a transformer together. The model sees every word of the question next to every word of the document and can reason about their relationship directly. The output is a single relevance score between zero and one. Much more accurate, much slower. You cannot pre-compute anything, so every query and document pair costs a full forward pass through the model.

Re-ranking combines the two. A bi-encoder narrows a million documents down to ten in milliseconds. A cross-encoder re-sorts those ten in about one second. You get almost all of the accuracy of a full cross-encoder search at a small fraction of the cost. That is the pipeline you are about to build.

## Version B: add Cohere Rerank

`rerank_with_cohere` takes the existing ten-chunk vector search result and sends the question plus every chunk's text to Cohere is hosted cross-encoder. Cohere returns a relevance score between zero and one for each chunk, where one is a perfect match. The helper sorts by relevance and returns the top three.

The code is three lines. One function call, one output shape, no special adapters. Behind the scenes Cohere is running `rerank-english-v3.0`, a production cross-encoder that has been fine tuned for exactly this task.

In [ ]:
from utils.shared import rerank_with_cohere, show_rerank_results

# top_n is how many of the ten input chunks Cohere returns, re-sorted by
# relevance. Three is the standard production default. Change it to 5 in the
# "Try it yourself" section below and watch the ordering shift.
top_n = 3

reranked_results = rerank_with_cohere(
    question=question,
    results=vector_results,
    top_n=top_n,
    model="rerank-english-v3.0",
)
show_rerank_results(reranked_results, question=question)

## What the rerank did

Compare this table to the vector-only table above.

The top chunk is now from the refund policy. The cross-encoder understood that a question about getting money back for a yearly purchase is really a question about how refunds are calculated, not about how to cancel or what the marketing team promised two years ago. The relevance score on the top chunk is close to one, which is the cross-encoder saying it is confident this chunk directly answers the question.

The correct pro-rated calculation chunk, which was buried at rank four in the vector search, now sits at rank two with a relevance score near 0.97. It is in the top three and the LLM is about to see it.

The outdated blog post may still show up at rank three because Cohere does not know the policy changed. It can only compare content to the question, not dates or document freshness. You are about to watch the LLM handle that. Two refund policy chunks dominate the top of the list, so the LLM has much stronger signal to anchor on than it did with the vector-only pipeline, even if a little noise from the stale blog post leaks through. The "Try it yourself" section below shows you how to squeeze the stale chunk out entirely by lowering `top_n` to two.

## The moment that matters, what the LLM actually says now

Feed the reranked top three to the same LLM and see what comes out.

In [ ]:
answer_reranked = generate_answer(reranked_results, question)
print(answer_reranked)

## Read both answers side by side

Scroll back up and read the vector-only answer first. It tells the customer they can get a thirty day full refund, no deductions, no processing fee. That is the stale 2024 policy and it is confidently wrong.

Now read the reranked answer. It explains that the refund is pro-rated, minus two months of service, minus a fifty dollar processing fee. That is the current Section 4.2 policy, the one the billing team actually enforces. One answer would trigger a customer chargeback when the real refund lands short. The other sets the correct expectation on the first touch.

One function call separated a confident wrong answer from a correct one. The retrieval improved from "pro-rated chunk sits at rank four" to "pro-rated chunk sits at rank two". The LLM, given the same prompt template and the same model, produced a different answer because the context changed.

The cross-encoder did not make the LLM smarter. It gave the LLM better upstream context. That is the whole job of reranking. If you ever see the reranked answer still mention the stale thirty day guarantee, that is the blog post chunk leaking in at rank three. Drop `top_n` to two and rerun. You will see the stale chunk pushed out of the context entirely and the answer land on the pro-rated formula only.

## Why not use a cross-encoder for everything

If cross-encoders are so much more accurate, why does anyone still use bi-encoders at retrieval time? The answer is math.

Imagine a corpus of one hundred thousand documents. A bi-encoder pre-computes every document embedding once and stores them. At query time, it encodes the question and does vector comparison. Total search time is under one hundred milliseconds.

A cross-encoder cannot pre-compute anything. Every query has to run against every document, as a fresh forward pass through the transformer. At around fifty milliseconds per pair, one hundred thousand documents is about eighty three minutes per search. No user waits eighty three minutes for a refund answer.

Re-ranking is the middle path. The bi-encoder narrows one hundred thousand documents to ten in milliseconds. The cross-encoder re-sorts those ten in about one second. Total latency is about one second, which is fine for a support chatbot. You get roughly ninety five percent of the accuracy of a full cross-encoder search at well under one percent of the cost.

The tradeoff is not "cross-encoder good, bi-encoder bad". The tradeoff is "narrow broadly with a cheap model, refine narrowly with an expensive one". The same pattern shows up in every ranking system, from search engines to recommendation pipelines to ad auctions. Chapter 5 is just one more place you see it.

In [ ]:
import time

# Time the vector search across ten candidates.
start = time.perf_counter()
_ = search(index, question, k=10)
vector_ms = (time.perf_counter() - start) * 1000

# Time the rerank step across the same ten candidates.
start = time.perf_counter()
_ = rerank_with_cohere(question=question, results=vector_results, top_n=3)
rerank_ms = (time.perf_counter() - start) * 1000

print(f"Vector search (k=10):     {vector_ms:7.1f} ms")
print(f"Cohere rerank (top_n=3):  {rerank_ms:7.1f} ms")
print(f"Total pipeline:           {vector_ms + rerank_ms:7.1f} ms")

The numbers you see will wobble from run to run, but the shape is stable. Vector search finishes in a few hundred milliseconds. Rerank takes about half a second to a second on top of that. Together you are under two seconds for a full retrieve then re-sort pipeline. That is fast enough for any human-facing chatbot and it is the price of the lift in answer quality you just saw.

## Re-ranking APIs you can reach for

You do not have to build a cross-encoder from scratch. Several hosted and open options exist, and they all fit the same two-stage pattern you just built.

- **Cohere Rerank.** The most popular hosted option and the one this chapter uses. Three lines of code. Free tier covers one thousand rerank calls per month, enough for an entire cohort of students plus a small production pilot. Model names include `rerank-english-v3.0` (English only) and `rerank-multilingual-v3.0` (one hundred plus languages).
- **Jina Reranker.** An open option with a generous free tier and open-source model weights if you want to self-host. Good choice when you want to keep everything on your own infrastructure without training a model.
- **Voyage AI.** A newer entrant focused on domain-specific reranking for legal, code, and financial text. Worth a look if your corpus is heavy on one of those domains.
- **Self-hosted cross-encoders on HuggingFace.** The `cross-encoder/ms-marco-MiniLM-L-12-v2` model from the `sentence-transformers` library is a common starting point. Runs on CPU for small workloads, fits on a small GPU for production. No API keys, no rate limits, but you own the latency and the ops. Chapter 5 homework walks you through this path.

The concept is identical across all of them. Retrieve broadly, then rerank precisely. Pick the one that fits your stack, your budget, and your data. The rest of the pipeline does not change.

## Open your LangSmith trace

Open [smith.langchain.com](https://smith.langchain.com), click into the project called `rag-for-pms`, and open the most recent trace. You will see the vector search, the Cohere rerank call, and both LLM calls in one visual waterfall. The rerank step shows up as its own span with input chunks, output chunks, and the relevance scores. Five minutes of clicking around the trace will make the before and after feel concrete in a way the printed tables cannot.

## What you can do at work on Monday

Reranking is the fix most production RAG systems reach for when they hit the ceiling of "the retrieval scores look fine but the answers are subtly wrong". Here are the three questions to ask on your next RAG review.

1. When your vector search returns the top three chunks, are those three the chunks a human reviewer would actually pick to answer the question? Pull ten real user questions from your logs and grade the retrieved chunks by hand. You will find disagreement much faster than you expect.
2. What is the oldest or most stale document in your corpus, and how often does it still show up in the top three for a current question? Stale docs with strong keyword overlap are the exact pattern rerank fixes. If your team has a "delete the old stuff" backlog, rerank can buy you time while you work through it.
3. Have you measured the latency and cost of adding a rerank step to the pipeline? One extra second and a few tenths of a cent per query is almost always worth a measurable quality lift, but your team should know the numbers, not guess.

A team that cannot answer these questions cleanly is probably leaving a significant quality gain on the table. Rerank is one of the cheapest high-impact changes you can make to a production RAG pipeline that is already indexing cleanly.

## Try it yourself

Pick any of these. Change the value in the relevant cell above, rerun, watch what happens.

1. Change `top_n=3` to `top_n=2` in the `rerank_with_cohere` call. Rerun the rerank cell and the reranked-answer cell. The outdated blog post chunk falls out of the context completely and the LLM answer lands on the pro-rated math without any mention of the stale thirty day refund. This is the cleanest version of the payoff.
2. Change the `question` variable to `"What is the exact dollar amount refunded if I cancel my 1499 dollar Pro Annual plan in month five?"` and rerun the vector search and the rerank cells. This is a much more specific phrasing. Watch how the vector search already lands the correct chunk at rank one, and how rerank barely changes anything. Specific questions do not need rerank. Vague ones do. This is the single most important intuition to carry out of the chapter.
3. Change the `question` to `"Tell me about the SkillAgents product guide"`. This question has nothing to do with refunds. Rerun the vector search, the rerank cell, and both answer cells. Notice the refund policy chunks drop out of the rerank entirely, because Cohere knows they do not answer the question. Rerank fails softly when there is nothing relevant to retrieve, which is the exact behavior you want in production.

## Homework

Two exercises you can do on your own. Each one takes about fifteen minutes.

1. **Try a self-hosted cross-encoder and measure the latency.** Install `sentence-transformers` locally and load `cross-encoder/ms-marco-MiniLM-L-12-v2`. Score the same ten vector search results against the same question. Print the latency for each call and the top three by the local model's score. Compare the latency to the Cohere latency you saw in this notebook. How much slower is the local cross-encoder on CPU? Is the quality of the top three the same? This is the closest you will get to a production reranker you own end to end, with no API dependencies.

2. **Apply reranking to your own company.** Pick three real user questions from your own support logs or your own product. Index your own documents with the same chunk size. Run each question through vector search first, then through `rerank_with_cohere`. For each question, print the top three before and after rerank and read both answers. Did rerank fix any of the three? Did it break any of the three? Share one example of each in the cohort Slack. This is the single best way to build intuition for when rerank helps and when it does not.

## What is next

In Chapter 6 your retrieval is clean, your query rewriter is grounded, your router picks the right source, your hybrid search catches the exact matches, and your reranker promotes the best chunks to the top. The pipeline looks great on the handful of questions you personally tested. And you still have no idea how the system performs on the long tail of real user questions you have never seen.

Chapter 6 is about evaluation. You will run a small test set through the full pipeline, score every answer on faithfulness, relevance, and context recall, and watch a pandas table light up green or red for every question. Then you will deliberately break the pipeline and watch the numbers fall so you know what a healthy pipeline looks like and what a broken one looks like. That is how you find regressions before your users do.

See you in Chapter 6.